<a href="https://colab.research.google.com/github/busycaesar/Finetune_LoRA/blob/Master/gemma_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import userdata

# Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
# vars as appropriate for your system.
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

In [2]:
%pip install -U -q keras-hub keras

In [3]:
os.environ["KERAS_BACKEND"] = "jax"  # Or "torch" or "tensorflow".
# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

In [4]:
import keras
import keras_hub

In [5]:
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_instruct_1b")

In [6]:
template = "Instruction:\n{instruction}\n\nResponse:\n{response}"

In [7]:
prompt = template.format(
    instruction="I wake in the night, usually about 2-3 hours after going to sleep, with both feet and legs to mid calf feeling like they are on fire. slight red discolorization, minor swelling. This is very painful but after getting up, I can walk it off in about 30 minutes.",
    response="",
)

print(gemma_lm.generate(prompt, max_length=500))

Instruction:
I wake in the night, usually about 2-3 hours after going to sleep, with both feet and legs to mid calf feeling like they are on fire. slight red discolorization, minor swelling. This is very painful but after getting up, I can walk it off in about 30 minutes.

Response:
This sounds like a case of **peripheral neuropathy**.

Here's why:

*   **Pain and Burning Sensation:** The described pain and burning sensation are consistent with nerve damage.
*   **Restricted Movement:** The feeling of "fire" in the legs and the difficulty walking are classic symptoms of peripheral neuropathy.
*   **Red Discoloration and Swelling:** These are often associated with nerve damage and inflammation.

**Important Disclaimer:** *I am an AI Chatbot and not a medical professional. This information is for general knowledge and informational purposes only, and does not constitute medical advice. It is essential to consult with a qualified healthcare professional for any health concerns or before m

In [9]:
from datasets import load_dataset

ds = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")

# Convert to pandas and sample
df = ds.to_pandas().sample(1000, random_state=42)

features = {
    "prompts": df["input"].tolist(),
    "responses": df["output"].tolist()
}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [10]:
print(len(features['prompts']))
print(features['prompts'][0])
print(features['responses'][0])

1000
I wake in the night, usually about 2-3 hours after going to sleep, with both feet and legs to mid calf feeling like they are on fire. slight red discolorization, minor swelling. This is very painful but after getting up, I can walk it off in about 30 minutes.
Dear patient Here are the possibilities of what you might have.1)PhlebitisPhlebitis means inflammation of the veins, and can cause redness, itching, irritation, pain, and swelling. A simple Doppler can rule this out.2Blood clot in the lifeblood clots in the leg can become very dangerous, symptoms include swelling, redness, tenderness in the leg. Coagulation profile with an angiography may be required3)Cellulitis


In [11]:
gemma_lm.backbone.enable_lora(rank=4)

In [12]:
# Limit the input sequence length to 256 (to control memory usage).
gemma_lm.preprocessor.sequence_length = 256
# Use AdamW (a common optimizer for transformer models).
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)
# Exclude layernorm and bias terms from decay.
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

In [13]:
gemma_lm.fit(features, epochs=1, batch_size=1)

1000/1000 ━━━━━━━━━━━━━━━━━━━━ 409s 374ms/step - loss: 1.5364 - sparse_categorical_accuracy: 0.3618


In [14]:
prompt = template.format(
    instruction="I wake in the night, usually about 2-3 hours after going to sleep, with both feet and legs to mid calf feeling like they are on fire. slight red discolorization, minor swelling. This is very painful but after getting up, I can walk it off in about 30 minutes.",
    response="",
)

print(gemma_lm.generate(prompt, max_length=500))

Instruction:
I wake in the night, usually about 2-3 hours after going to sleep, with both feet and legs to mid calf feeling like they are on fire. slight red discolorization, minor swelling. This is very painful but after getting up, I can walk it off in about 30 minutes.

Response:
The symptoms you are describing are likely due to a condition called "foot pain syndrome" or "foot pain syndrome." It is a common condition that can be caused by a variety of factors, including:

1.  **Overuse:** This is the most common cause. It is caused by repetitive stress on the foot, which can be caused by running, jumping, or other high-impact activities.
2.  **Poor Footwear:** Wearing shoes that don't fit properly or that don't provide adequate support can contribute to foot pain.
3.  **Flat Feet:** Flat feet can cause the arch of the foot to collapse, which can lead to pain in the heel and arch.
4.  **Bunions:** A bunion is a bony bump on the bottom of the foot.
5.  **Hammer Toe:** A hammer toe is 

In [20]:
gemma_lm.backbone.save_lora_weights("lora_weights.lora.h5")